# range_ladder — Optuna walk-forward (Phase A / A.1 / A.2)

Tunes the ladder structure of the live `range_inventory_ladder` controller
with a 3-fold walk-forward on lake candles. Timing parameters stay FROZEN at
live values (Phase B owns them).

Two search modes (`SEARCH_MODE`):
- **generative** — the 10-param generative family; rungs rebuilt per fold at
  the train-only median-3 anchor; export rebuilds at the DEPLOY anchor.
- **refine_incumbent** — refines the live ladder itself: stage 1 overlay
  (shift/stretch/tilt, TPE) + optional stage 2 per-rung nudge (CMA-ES).
  The identity overlay is enqueued as trial 0, so the study can never lose
  to the incumbent baseline.

Gate policy (`GATE_MODE`): strict (Phase A endinv gate) | accumulate_ok
(endinv waived; conservative-floor + max-DD risk gates instead) | soft
(endinv becomes a score penalty). Fill-frequency preference (Phase A.2):
per-rung train-touch floor, trades/month and per-side fill gates, and an
optional consistency-blended objective (`OBJECTIVE_MODE`).

The incumbent benchmark runs through the SAME evaluator under the SAME
policy/objective as the trials.

```bash
papermill range_ladder_optuna_walkforward.ipynb out_xmr_a2.ipynb \
  -p CONNECTOR nonkyc -p TRADING_PAIR XMR-USDT -p N_TRIALS 600 \
  -p GATE_MODE accumulate_ok -p OBJECTIVE_MODE consistency
papermill range_ladder_optuna_walkforward.ipynb out_xmr_refine_a2.ipynb \
  -p CONNECTOR nonkyc -p TRADING_PAIR XMR-USDT -p SEARCH_MODE refine_incumbent \
  -p GATE_MODE accumulate_ok -p OBJECTIVE_MODE consistency
```

In [1]:
# Bootstrap: make pmm_lab importable, discover the subproject root, load .env.
import os
import sys
from pathlib import Path


def _find_subproject_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pmm_lab" / "__init__.py").exists():
            return base
    raise RuntimeError(f"pmm_dynamic subproject root not found above {Path.cwd()}")


SUBPROJECT_ROOT = _find_subproject_root()
if str(SUBPROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(SUBPROJECT_ROOT))

# .env discovery (MONGO_URI, OPTUNA_STORAGE) — walk up from the subproject root
_d = SUBPROJECT_ROOT
for _ in range(10):
    _env = _d / ".env"
    if _env.exists():
        for _line in _env.read_text(encoding="utf-8").splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())
        break
    _d = _d.parent

print(f"subproject root: {SUBPROJECT_ROOT}")

subproject root: /quants-lab/research_notebooks/market_lab/pmm_dynamic


In [2]:
# Parameters
CONNECTOR = "nonkyc"            # "nonkyc" | "kraken"
TRADING_PAIR = "XMR-USDT"       # nonkyc: XMR/DASH/SUN/ZANO-USDT; kraken: XMR-USDT, XMR-USD
INTERVAL = "1h"
SEARCH_MODE = "generative"      # "generative" | "refine_incumbent"
N_TRIALS = 1000
N_STARTUP_TRIALS = 200
REFINE_STAGE2 = True            # refine mode: per-rung CMA-ES nudge after the overlay
N_TRIALS_STAGE2 = 400
INCUMBENT_TRIAL = True          # benchmark + warm-start from a live YAML when present
FUND_USD = 1000.0               # deployed fund (quote units) — NOT tuned
QUOTE_FRAC = 0.5
N_JOBS = 1                      # >1 requires PostgreSQL OPTUNA_STORAGE
RUN_STRESS = True               # per-fold conservative re-score
STRESS_SPREAD_PCT = 0.0         # measured spread; stress slip = max(0.001, spread/2)
SEED = 12345
MIN_USABLE_DAYS = 150.0
MAX_GAP_PCT = 5.0
INCUMBENTS_DIR = "configs/incumbents"   # relative → anchored to the subproject root
ARTIFACTS_DIR = "artifacts/range_ladder"

# --- Gate policy (Phase A.1 §2) ---
GATE_MODE = "strict"            # "strict" | "accumulate_ok" | "soft"
ENDINV_GATE_PCT = 75.0
ENDINV_PENALTY = 20.0           # soft mode: score penalty scale
CONS_FLOOR_ANN_PCT = 0.0        # accumulate_ok: conservative annualized floor
MAX_DD_PCT = 60.0               # accumulate_ok: fold max-drawdown ceiling

# --- Fill-frequency preference (Phase A.2 §3) ---
MIN_RUNG_TOUCHES_TRAIN = 8      # per-rung train touches; 0 disables
TOUCH_LOOKBACK_DAYS = 270.0
MIN_TRADES_PER_MONTH = 6.0      # fold gate; 0 disables
MIN_SIDE_FILLS_PER_FOLD = 3     # fold gate; 1 = plain two-sidedness
OBJECTIVE_MODE = "median_ann"   # "median_ann" | "consistency"

## 1. Environment + storage preflight

In [3]:
import logging

import numpy as np
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print("MONGO_URI:", "SET" if MONGO_URI else "NOT SET")
print("OPTUNA_STORAGE:", "SET" if OPTUNA_STORAGE else "NOT SET (SQLite fallback)")

from pmm_lab.optuna.storage import get_storage_url

_storage_url = OPTUNA_STORAGE if OPTUNA_STORAGE else get_storage_url()
_is_pg = "postgresql" in _storage_url.lower()
if N_JOBS > 1 and not _is_pg:
    print(f"[preflight] N_JOBS={N_JOBS} with non-PostgreSQL storage -> forcing serial (N_JOBS=1)")
    N_JOBS = 1
print(f"[preflight] dispatch: {'process-parallel (PostgreSQL)' if N_JOBS > 1 else 'serial'}")

MONGO_URI: SET
OPTUNA_STORAGE: SET
[preflight] dispatch: serial


## 2. Data preflight (§5) + deploy anchor + gate policy

The DEPLOY anchor (median of the last 3 closes) drives feasibility checks
and the export — never the full-history median (Phase A.1 §1). Per-fold
trial anchoring stays train-only median-3 inside the objective.

In [4]:
from pmm_lab.config.defaults import INTERVAL_SECONDS
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.data.coverage import audit_pair, preflight_pair
from pmm_lab.data.hashing import hash_candles
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.optuna.objective_wrapper_range_ladder import (
    GatePolicy, plan_range_ladder_folds,
)
from pmm_lab.strategies.range_ladder import compute_anchor

loader = MongoCandleLoader()

print(f"=== Data preflight: {CONNECTOR} {TRADING_PAIR} ===")
for _iv in ("5m", INTERVAL):
    _r = audit_pair(CONNECTOR, TRADING_PAIR, _iv, loader=loader)
    print(f"  {_r['interval']:>4}: bars={_r['bars']:>8} days={_r['days']:8.1f} "
          f"gap_pct={_r['gap_pct']:6.2f}% max_gap_h={_r['max_gap_hours']:8.1f}")

candles, preflight_info = preflight_pair(
    CONNECTOR, TRADING_PAIR, interval=INTERVAL, loader=loader,
    min_usable_days=MIN_USABLE_DAYS, max_gap_pct=MAX_GAP_PCT,
)
print(f"source: {preflight_info['source']}, {len(candles)} bars")

bar_interval_seconds = INTERVAL_SECONDS[INTERVAL]
dataset_hash = hash_candles(candles)
rules_db = load_exchange_rules()
pair_rules = resolve_pair_rules(rules_db, CONNECTOR, TRADING_PAIR)
maker_fee = pair_rules.fees.maker_fee
cooldown_bars = max(0, round(3600 / bar_interval_seconds))

# Deploy anchor (Phase A.1 §1): feasibility + export price basis.
deploy_anchor = compute_anchor(candles["close"])
last_close = float(candles["close"][-1])
anchor_divergence_pct = abs(deploy_anchor - last_close) / last_close * 100.0
hist_median = float(np.median(candles["close"]))
reference_price = deploy_anchor
print(f"deploy_anchor={deploy_anchor:.6g} (last close {last_close:.6g}, "
      f"divergence {anchor_divergence_pct:.2f}%; full-history median "
      f"{hist_median:.6g} — informational only)")
if anchor_divergence_pct > 2.0:
    print(f"!! WARNING: deploy anchor diverges {anchor_divergence_pct:.1f}% from "
          f"last close -> trending, not ranging. Re-check before deploying.")

gate_policy = GatePolicy(
    mode=GATE_MODE,
    endinv_gate_pct=ENDINV_GATE_PCT,
    endinv_penalty=ENDINV_PENALTY,
    cons_floor_ann_pct=CONS_FLOOR_ANN_PCT,
    max_dd_pct=MAX_DD_PCT,
    min_trades_per_month=MIN_TRADES_PER_MONTH,
    min_side_fills_per_fold=MIN_SIDE_FILLS_PER_FOLD,
    min_rung_touches_train=MIN_RUNG_TOUCHES_TRAIN,
    touch_lookback_days=TOUCH_LOOKBACK_DAYS,
)
print(f"gate policy: {gate_policy.describe()}")
print(f"objective mode: {OBJECTIVE_MODE}")

train_days, test_days, step_days = plan_range_ladder_folds(len(candles), bar_interval_seconds)
print(f"fold plan: train={train_days:.1f}d, test={test_days:.1f}d x 3 folds (step={step_days:.1f}d)")
print(f"maker_fee={maker_fee} (dead-zone floor {4 * maker_fee:.4f}), cooldown_bars={cooldown_bars}")
print(f"dataset_hash={dataset_hash[:16]}...")

=== Data preflight: nonkyc XMR-USDT ===
    5m: bars=  326776 days=  1134.8 gap_pct=  0.02% max_gap_h=     4.3
    1h: bars=   27858 days=  1160.7 gap_pct=  0.00% max_gap_h=     0.0
source: native, 27858 bars
deploy_anchor=326.11 (last close 328.28, divergence 0.66%; full-history median 178.475 — informational only)
gate policy: {'mode': 'strict', 'endinv_gate_pct': 75.0, 'endinv_penalty': 20.0, 'cons_floor_ann_pct': 0.0, 'max_dd_pct': 60.0, 'min_trades_per_month': 6.0, 'min_side_fills_per_fold': 3, 'min_rung_touches_train': 8, 'touch_lookback_days': 270.0}
objective mode: median_ann
fold plan: train=980.8d, test=60.0d x 3 folds (step=60.0d)
maker_fee=0.002 (dead-zone floor 0.0080), cooldown_bars=1
dataset_hash=af7118f9c5a724cb...


## 3. Incumbent benchmark (§3.7)

Evaluated through the SAME fold machinery, gate policy, and objective mode
as the trials. Touch counts are reported but never gate the incumbent.
`refine_incumbent` mode ABORTS here when no incumbent YAML exists.

In [5]:
from pmm_lab.export.hb_yaml_range_ladder import (
    incumbent_yaml_path, load_range_ladder_incumbent,
)
from pmm_lab.objective.walkforward import TimeSeriesCV
from pmm_lab.optuna.objective_wrapper_range_ladder import (
    evaluate_ladder_walkforward,
)
from pmm_lab.strategies.range_ladder import RangeLadderConfig
from pmm_lab.strategies.range_ladder_gen import (
    fit_generative_to_ladder, ladder_round_trip_error,
)

incumbent = None
incumbent_fit_params = None
incumbent_summary = None

_cv = TimeSeriesCV(
    n_bars=len(candles), bar_interval_seconds=bar_interval_seconds,
    train_days=train_days, test_days=test_days, step_days=step_days,
    embargo_bars=0, macd_slow=3, natr_length=3,
)
fold_defs = _cv.get_folds()

if INCUMBENT_TRIAL or SEARCH_MODE == "refine_incumbent":
    # A relative INCUMBENTS_DIR is anchored to the subproject root (the
    # kernel CWD is usually notebooks/range_ladder, NOT the repo).
    _dir = Path(INCUMBENTS_DIR) if INCUMBENTS_DIR else Path("configs/incumbents")
    if not _dir.is_absolute():
        _dir = SUBPROJECT_ROOT / _dir
    _path = incumbent_yaml_path(_dir, CONNECTOR, TRADING_PAIR)
    incumbent = load_range_ladder_incumbent(_path)
    if incumbent is None:
        if SEARCH_MODE == "refine_incumbent":
            raise RuntimeError(
                f"SEARCH_MODE=refine_incumbent requires a live incumbent YAML at "
                f"{_path} — none found. Copy the live controller config there "
                f"(see configs/incumbents/README.md) or use SEARCH_MODE=generative."
            )
        print(f"no incumbent for ({CONNECTOR}, {TRADING_PAIR}) at {_path} — proceeding without a benchmark")

if incumbent is not None:
    lit_config = RangeLadderConfig(
        fund_quote=FUND_USD, quote_frac=QUOTE_FRAC, fee=maker_fee,
        cooldown_bars=cooldown_bars, stress_spread_pct=STRESS_SPREAD_PCT,
        literal_buy_prices=tuple(incumbent["buy_prices"]),
        literal_buy_weights=tuple(incumbent["buy_weights"]),
        literal_sell_prices=tuple(incumbent["sell_prices"]),
        literal_sell_weights=tuple(incumbent["sell_weights"]),
    )
    _rungs = lit_config.resolve_rungs(deploy_anchor, pair_rules.price_tick)
    incumbent_result = evaluate_ladder_walkforward(
        candles, fold_defs, bar_interval_seconds,
        rung_provider=lambda fd: (_rungs, deploy_anchor, None),
        fund=FUND_USD, quote_frac=QUOTE_FRAC, fee=maker_fee,
        cooldown_bars=cooldown_bars,
        stress_config=lit_config, run_stress=RUN_STRESS,
        gate_policy=gate_policy, objective_mode=OBJECTIVE_MODE,
        trial=None,
    )
    incumbent_summary = dict(
        objective=incumbent_result["objective"],
        folds=incumbent_result["fold_detail"],
        violations=incumbent_result["violations"],
        result=incumbent_result,
    )
    print(f"=== Incumbent benchmark (policy={GATE_MODE}, objective={OBJECTIVE_MODE}) ===")
    for _r in incumbent_result["fold_detail"]:
        _gate = f" GATE[{'; '.join(_r['gate_reasons'])}]" if _r.get("gated") else ""
        _cons = f"{_r['cons_ann_pct']:8.1f}%" if _r.get("cons_ann_pct") is not None else "     n/a"
        print(f"  fold {_r['fold']}: score={_r['score_ann_pct']:8.1f}  ann={_r['ann_pnl_pct']:8.1f}%  "
              f"cons={_cons}  endinv={_r['endinv_pct']:5.1f}%  "
              f"fills={_r['buy_fills']}b/{_r['sell_fills']}s  tpm={_r['trades_per_month']:.1f}{_gate}")
        if _r.get("rung_touches"):
            print(f"           train touches: buys={_r['rung_touches']['buys']} sells={_r['rung_touches']['sells']}")
    print(f"  incumbent objective: {incumbent_summary['objective']:.2f} "
          f"(gate violations {incumbent_summary['violations']}/{len(fold_defs)} — reported, never pruned)")

    if SEARCH_MODE == "generative":
        incumbent_fit_params = fit_generative_to_ladder(
            incumbent["buy_prices"], incumbent["buy_weights"],
            incumbent["sell_prices"], incumbent["sell_weights"],
        )
        _rt = ladder_round_trip_error(
            incumbent["buy_prices"], incumbent["buy_weights"],
            incumbent["sell_prices"], incumbent["sell_weights"], pair_rules.price_tick,
        )
        incumbent_fit_params.pop("anchor")
        print(f"generative approximation (round-trip err {_rt * 100:.2f}%): {incumbent_fit_params}")

=== Incumbent benchmark (policy=strict, objective=median_ann) ===
  fold 0: score=    65.4  ann=    65.4%  cons=    56.2%  endinv=  2.4%  fills=96b/125s  tpm=112.0
           train touches: buys=[304, 416, 358, 366, 434, 354, 241] sells=[266, 286, 290, 219, 107, 74, 73, 75, 87]
  fold 1: score=    54.5  ann=    54.5%  cons=    50.0%  endinv=  4.9%  fills=67b/65s  tpm=66.9
           train touches: buys=[323, 412, 364, 353, 416, 344, 241] sells=[280, 254, 277, 200, 134, 86, 79, 88, 105]
  fold 2: score=    76.6  ann=    76.6%  cons=    73.4%  endinv= 98.6%  fills=64b/68s  tpm=66.9 GATE[endinv 98.6% > 75%]
           train touches: buys=[321, 273, 191, 176, 166, 175, 158] sells=[271, 257, 365, 348, 265, 176, 134, 148, 188]
  incumbent objective: 59.96 (gate violations 1/3 — reported, never pruned)
generative approximation (round-trip err 274.27%): {'n_buy': 7, 'n_sell': 9, 'buy_near_pct': 0.00530705079605762, 'buy_far_pct': 0.07505686125852919, 'sell_near_pct': 0.00530705079605762, 'sell

## 4. Optuna study

generative: TPE + HyperbandPruner, incumbent approximation enqueued.
refine_incumbent: stage 1 overlay (TPE, identity enqueued as trial 0) +
optional stage 2 per-rung nudge (CMA-ES, identity enqueued) — the winner is
accepted only if it beats BOTH the stage-1 winner and the incumbent.

In [6]:
from pmm_lab.optuna.notebook_dispatch import optimize_study_for_notebook
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.objective_wrapper_range_ladder import (
    create_range_ladder_refine_objective,
)
from pmm_lab.optuna.search_space_range_ladder import (
    IDENTITY_OVERLAY_PARAMS, identity_nudge_params,
)
from pmm_lab.optuna.study import create_study

pruner = optuna.pruners.HyperbandPruner(min_resource=1, max_resource=3, reduction_factor=3)
refine_stage1_best = None
refine_stage2_best = None

if SEARCH_MODE == "generative":
    study_name = (f"{CONNECTOR}_{TRADING_PAIR}_{INTERVAL}_range_ladder_v2"
                  f"_{GATE_MODE}_{OBJECTIVE_MODE}")
    study = create_study(
        study_name=study_name, storage_url=_storage_url, seed=SEED,
        n_startup_trials=N_STARTUP_TRIALS, pruner=pruner,
    )
    if incumbent_fit_params is not None and len(study.trials) == 0:
        study.enqueue_trial(incumbent_fit_params)
        print("enqueued incumbent generative approximation as trial 0")

    factory_kwargs = dict(
        candles=candles,
        pair_rules=pair_rules,
        bar_interval_seconds=bar_interval_seconds,
        dataset_hash=dataset_hash,
        reference_price=reference_price,   # deploy anchor (Phase A.1 §1)
        strategy_name="range_ladder",
        train_days=train_days, test_days=test_days, step_days=step_days,
        run_stress=RUN_STRESS,
        fixed_quote=FUND_USD,
        objective_version=2,
        gate_policy=gate_policy,
        objective_mode=OBJECTIVE_MODE,
    )
    if STRESS_SPREAD_PCT:
        from dataclasses import replace as _replace
        from pmm_lab.optuna.canonicalizer_range_ladder import canonicalize_range_ladder_params

        def _canon_with_spread(raw, pr, ref, bar_interval_seconds=3600):
            bundle, reason = canonicalize_range_ladder_params(
                raw, pr, ref, bar_interval_seconds=bar_interval_seconds)
            if bundle is not None:
                bundle.strategy_config = _replace(
                    bundle.strategy_config, stress_spread_pct=STRESS_SPREAD_PCT)
            return bundle, reason

        factory_kwargs["strategy_canonicalizer"] = _canon_with_spread
        N_JOBS = 1

    study = optimize_study_for_notebook(
        study_name=study_name,
        storage_url=_storage_url,
        n_trials=N_TRIALS,
        n_jobs=N_JOBS,
        objective_factory=create_objective,
        factory_kwargs=factory_kwargs,
        sampler_seed=SEED,
        n_startup_trials=N_STARTUP_TRIALS,
        pruner=pruner,
    )

elif SEARCH_MODE == "refine_incumbent":
    _incumbent_rungs = {
        "buy_prices": incumbent["buy_prices"],
        "buy_weights": incumbent["buy_weights"],
        "sell_prices": incumbent["sell_prices"],
        "sell_weights": incumbent["sell_weights"],
    }
    _refine_common = dict(
        candles=candles,
        pair_rules=pair_rules,
        bar_interval_seconds=bar_interval_seconds,
        dataset_hash=dataset_hash,
        deploy_anchor=deploy_anchor,
        train_days=train_days, test_days=test_days, step_days=step_days,
        fund=FUND_USD, quote_frac=QUOTE_FRAC, cooldown_bars=cooldown_bars,
        stress_spread_pct=STRESS_SPREAD_PCT, run_stress=RUN_STRESS,
        gate_policy=gate_policy, objective_mode=OBJECTIVE_MODE,
    )

    # ---- Stage 1: overlay search ----
    study_name = (f"{CONNECTOR}_{TRADING_PAIR}_{INTERVAL}_range_ladder_refine1_v2"
                  f"_{GATE_MODE}_{OBJECTIVE_MODE}")
    study = create_study(
        study_name=study_name, storage_url=_storage_url, seed=SEED,
        n_startup_trials=N_STARTUP_TRIALS, pruner=pruner,
    )
    if len(study.trials) == 0:
        study.enqueue_trial(dict(IDENTITY_OVERLAY_PARAMS))
        print("enqueued IDENTITY overlay as trial 0 (bit-for-bit incumbent)")
    study = optimize_study_for_notebook(
        study_name=study_name,
        storage_url=_storage_url,
        n_trials=N_TRIALS,
        n_jobs=N_JOBS,
        objective_factory=create_range_ladder_refine_objective,
        factory_kwargs=dict(_refine_common, base_rungs=_incumbent_rungs, stage="overlay"),
        sampler_seed=SEED,
        n_startup_trials=N_STARTUP_TRIALS,
        pruner=pruner,
    )
    _s1_completed = [t for t in study.trials
                     if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None]
    if not _s1_completed:
        raise RuntimeError(
            "refine stage 1: no completed trials — even the identity overlay was "
            "pruned, i.e. the INCUMBENT ITSELF fails the configured gate policy "
            f"({GATE_MODE}) and/or the train-touch floor "
            f"(MIN_RUNG_TOUCHES_TRAIN={MIN_RUNG_TOUCHES_TRAIN}) on this data. "
            "An incumbent with never-touched deep rungs needs "
            "MIN_RUNG_TOUCHES_TRAIN=0 to be refinable. "
            "Review the incumbent benchmark above."
        )
    refine_stage1_best = max(_s1_completed, key=lambda t: t.value)
    print(f"stage 1 winner: trial #{refine_stage1_best.number} "
          f"objective={refine_stage1_best.value:.2f} params={refine_stage1_best.params}")

    # ---- Stage 2: per-rung CMA-ES nudge around the stage-1 winner ----
    if REFINE_STAGE2:
        try:
            import cmaes  # noqa: F401
        except ImportError:
            raise RuntimeError(
                "REFINE_STAGE2 requires the `cmaes` package for optuna's "
                "CmaEsSampler — pip install cmaes (in this kernel's env)."
            )
        _s1_rungs = refine_stage1_best.user_attrs["refined_rungs"]
        _stage2_base = {
            "buy_prices": _s1_rungs["buys"],
            "buy_weights": _s1_rungs["buy_weights"],
            "sell_prices": _s1_rungs["sells"],
            "sell_weights": _s1_rungs["sell_weights"],
        }
        _n_buy, _n_sell = len(_s1_rungs["buys"]), len(_s1_rungs["sells"])
        study2_name = (f"{CONNECTOR}_{TRADING_PAIR}_{INTERVAL}_range_ladder_refine2_v2"
                       f"_{GATE_MODE}_{OBJECTIVE_MODE}")
        _sampler2 = optuna.samplers.CmaEsSampler(seed=SEED, n_startup_trials=1)
        study2 = optuna.create_study(
            study_name=study2_name, storage=_storage_url,
            direction="maximize", load_if_exists=True,
            sampler=_sampler2, pruner=pruner,
        )
        if len(study2.trials) == 0:
            study2.enqueue_trial(identity_nudge_params(_n_buy, _n_sell))
            print("enqueued IDENTITY nudge as stage-2 trial 0 (== stage-1 winner)")
        _objective2 = create_range_ladder_refine_objective(
            **dict(_refine_common, base_rungs=_stage2_base, stage="nudge")
        )
        study2.optimize(_objective2, n_trials=N_TRIALS_STAGE2, catch=(Exception,))
        _s2_completed = [t for t in study2.trials
                         if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None]
        if _s2_completed:
            refine_stage2_best = max(_s2_completed, key=lambda t: t.value)
            print(f"stage 2 winner: trial #{refine_stage2_best.number} "
                  f"objective={refine_stage2_best.value:.2f}")
        else:
            print("stage 2: no completed trials — keeping the stage-1 winner")
else:
    raise ValueError(f"unknown SEARCH_MODE {SEARCH_MODE!r}")

[preflight] Phase 1 dispatch: serial (n_jobs=1)


## 5. Results

In [7]:
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
failed = [t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]
ranked = sorted(
    [t for t in completed if t.value is not None],
    key=lambda t: t.value, reverse=True,
)
print(f"trials: {len(study.trials)} total | {len(completed)} complete | "
      f"{len(pruned)} pruned | {len(failed)} failed")
if not ranked:
    raise RuntimeError(
        "NO COMPLETED TRIALS — every trial was pruned/rejected. "
        "Check the preflight audit, gate policy, and constraint floors."
    )

print(f"\n=== Top 10 of {len(ranked)} completed trials "
      f"(policy={GATE_MODE}, objective={OBJECTIVE_MODE}) ===")
print(f"{'#':>5} {'objective':>10} {'pnl_med%':>9} {'endinv_med%':>11} "
      f"{'cons_med':>9} {'tpm_med':>8} {'touch_min':>9} {'viol':>4}")
for _t in ranked[:10]:
    _ua = _t.user_attrs
    _tm = _ua.get("trades_per_month_median")
    _mt = _ua.get("min_rung_touches")
    print(f"{_t.number:>5} {_t.value:>10.2f} "
          f"{_ua.get('pnl_pct_median', float('nan')):>9.2f} "
          f"{_ua.get('endinv_pct_median', float('nan')):>11.1f} "
          f"{_ua.get('cons_score_median', float('nan')):>9.1f} "
          f"{(_tm if _tm is not None else float('nan')):>8.1f} "
          f"{(str(_mt) if _mt is not None else 'n/a'):>9} "
          f"{_ua.get('gate_violations', '?'):>4}")
if incumbent_summary is not None:
    print(f"{'INC':>5} {incumbent_summary['objective']:>10.2f}   "
          f"(live incumbent, same policy/objective; "
          f"violations {incumbent_summary['violations']})")

best = ranked[0]

# refine mode: acceptance rule — the champion must beat stage 1 AND the incumbent
refined_champion = None
if SEARCH_MODE == "refine_incumbent":
    _inc_obj = incumbent_summary["objective"]
    _s1 = refine_stage1_best
    _s2 = refine_stage2_best
    if (_s2 is not None and _s2.value > _s1.value and _s2.value > _inc_obj):
        refined_champion = {"trial": _s2, "stage": "stage2",
                            "rungs": _s2.user_attrs["refined_rungs"]}
    elif _s1.value > _inc_obj:
        refined_champion = {"trial": _s1, "stage": "stage1",
                            "rungs": _s1.user_attrs["refined_rungs"]}
    if refined_champion is not None:
        _t = refined_champion["trial"]
        print(f"\nrefined champion: {refined_champion['stage']} trial #{_t.number} "
              f"objective={_t.value:.2f} vs incumbent {_inc_obj:.2f} "
              f"(+{_t.value - _inc_obj:.2f})")
        best = _t
    else:
        print(f"\nINCUMBENT STANDS: no refinement beat the incumbent objective "
              f"({_inc_obj:.2f}) — stage1 {_s1.value:.2f}"
              + (f", stage2 {_s2.value:.2f}" if _s2 is not None else ""))

print(f"\nbest trial #{best.number}: objective={best.value:.2f}")
for _k, _v in sorted(best.params.items()):
    print(f"  {_k} = {_v}")

trials: 2000 total | 48 complete | 1952 pruned | 0 failed

=== Top 10 of 48 completed trials (policy=strict, objective=median_ann) ===
    #  objective  pnl_med% endinv_med%  cons_med  tpm_med touch_min viol
  878      63.91     10.67        74.1      58.3      nan       n/a    1
  674      54.86      9.26        70.1      20.7      nan       n/a    1
  897      45.88      8.10         9.2      40.6      nan       n/a    1
 1334      39.31      9.46        67.9      50.0     35.5        11    1
  614      24.37      5.05        51.1      29.6      nan       n/a    1
  496      19.90      5.24        28.6     -18.5      nan       n/a    1
  651      19.70      5.02        19.7       6.9      nan       n/a    1
  498       9.82      4.51        11.4      22.1      nan       n/a    1
  892       4.37      4.04        61.5      17.3      nan       n/a    1
  546       3.29      2.08        41.4      11.5      nan       n/a    1
  INC      59.96   (live incumbent, same policy/objective; vio

## 6. Export

Rungs are ALWAYS rebuilt from the winning trial's generative params at the
deploy anchor via the same build/constraint path (Phase A.2 §1); refined
ladders export their absolute rungs literally. The market-bracket validator
hard-fails an export that does not bracket the deploy anchor — the notebook
then refuses to keep the YAML (the report still records the failure).

In [8]:
from pmm_lab.export.hb_yaml_range_ladder import (
    RangeLadderExportParams, export_range_ladder_yaml,
)
from pmm_lab.export.validate_export import validate_yaml_file
from pmm_lab.optuna.canonicalizer_range_ladder import (
    canonicalize_range_ladder_params, effective_min_order_quote,
)
from pmm_lab.strategies.range_ladder_gen import validate_rungs

export_failed_reason = None
export_rungs = None
yaml_path = None
vr = None

_policy_comment = [
    f"Selected under gate policy: {gate_policy.describe()}",
    f"objective_mode: {OBJECTIVE_MODE}; search_mode: {SEARCH_MODE}",
    f"fill-frequency constraints: min_rung_touches_train={MIN_RUNG_TOUCHES_TRAIN} "
    f"(lookback {TOUCH_LOOKBACK_DAYS:g}d), min_trades_per_month={MIN_TRADES_PER_MONTH:g}, "
    f"min_side_fills_per_fold={MIN_SIDE_FILLS_PER_FOLD}",
]
if GATE_MODE == "accumulate_ok":
    _policy_comment.append(
        "NOTE: tuned under accumulate_ok — the endinv gate was WAIVED; this "
        "config may accumulate base inventory in declines by design.")
if anchor_divergence_pct > 2.0:
    _policy_comment.append(
        f"WARNING: deploy anchor {deploy_anchor:.6g} diverged "
        f"{anchor_divergence_pct:.1f}% from last close {last_close:.6g} at export "
        f"time (trending, not ranging).")

if SEARCH_MODE == "generative":
    raw_best = dict(best.params)
    raw_best["fund_quote"] = FUND_USD
    raw_best["quote_frac"] = QUOTE_FRAC
    raw_best["cooldown_time"] = 3600
    bundle, reason = canonicalize_range_ladder_params(
        raw_best, pair_rules, deploy_anchor, bar_interval_seconds=bar_interval_seconds)
    if bundle is None:
        export_failed_reason = f"best trial re-canonicalization failed at deploy anchor: {reason}"
    else:
        export_config = bundle.strategy_config
        export_rungs = export_config.resolve_rungs(deploy_anchor, pair_rules.price_tick)
        _near = {"buy_near_pct": export_config.buy_near_pct,
                 "sell_near_pct": export_config.sell_near_pct}
else:
    _r = refined_champion["rungs"] if refined_champion is not None else None
    if _r is None:
        export_failed_reason = "incumbent stands — no refined export written (live config unchanged)"
    else:
        from pmm_lab.strategies.range_ladder import RangeLadderConfig as _RLC
        export_config = _RLC(
            fund_quote=FUND_USD, quote_frac=QUOTE_FRAC, fee=maker_fee,
            cooldown_bars=cooldown_bars,
            literal_buy_prices=tuple(_r["buys"]),
            literal_buy_weights=tuple(_r["buy_weights"]),
            literal_sell_prices=tuple(_r["sells"]),
            literal_sell_weights=tuple(_r["sell_weights"]),
        )
        export_rungs = export_config.resolve_rungs(deploy_anchor, pair_rules.price_tick)
        _near = {}
        _policy_comment.append(
            f"refine_incumbent: {refined_champion['stage']} winner "
            f"(objective {refined_champion['trial'].value:.2f} vs incumbent "
            f"{incumbent_summary['objective']:.2f}); params: "
            f"{refined_champion['trial'].params}")

if export_failed_reason is None:
    ok, reason = validate_rungs(
        export_rungs, anchor=deploy_anchor, fee=maker_fee,
        price_tick=pair_rules.price_tick,
        min_order_quote=effective_min_order_quote(pair_rules, deploy_anchor),
        fund=FUND_USD, quote_frac=QUOTE_FRAC, **_near,
    )
    if not ok:
        export_failed_reason = f"deploy-anchor constraint check failed: {reason}"

if export_failed_reason is None:
    out_dir = Path(ARTIFACTS_DIR) / CONNECTOR
    yaml_path = out_dir / f"{TRADING_PAIR}_{INTERVAL}_screening_best.yml"
    export_range_ladder_yaml(
        export_config, deploy_anchor, pair_rules,
        RangeLadderExportParams(connector_name=CONNECTOR, trading_pair=TRADING_PAIR),
        yaml_path, total_amount_quote=FUND_USD,
        extra_comment_lines=_policy_comment,
    )
    vr = validate_yaml_file(
        str(yaml_path), price_tick=pair_rules.price_tick,
        deploy_anchor=deploy_anchor, **_near,
    )
    for _w in vr.warnings:
        print(f"  export warning: {_w}")
    if not vr.valid:
        export_failed_reason = f"export validation FAILED: {vr.errors}"
        yaml_path.unlink()
        yaml_path = None
        print(f"!! export REFUSED — YAML deleted: {export_failed_reason}")
    else:
        print(f"exported: {yaml_path}  valid=True")
        print(f"  buys:  {[float(x) for x in export_rungs.buys]}")
        print(f"  sells: {[float(x) for x in export_rungs.sells]}")
else:
    out_dir = Path(ARTIFACTS_DIR) / CONNECTOR
    print(f"!! no export written: {export_failed_reason}")

exported: artifacts/range_ladder/nonkyc/XMR-USDT_1h_screening_best.yml  valid=True
  buys:  [324.21, 245.53, 207.29]
  sells: [330.19, 345.1, 375.03]


## 7. Report

In [9]:
import json
from datetime import datetime, timezone

_ua = best.user_attrs
_fold_detail = _ua.get("fold_detail", [])

_lines = [
    f"# range_ladder report — {CONNECTOR} {TRADING_PAIR} {INTERVAL} ({SEARCH_MODE})",
    "",
    f"- Generated: {datetime.now(timezone.utc).isoformat()}",
    f"- Study: `{study.study_name}`",
    f"- Dataset: {len(candles)} bars ({preflight_info['source']}), hash `{dataset_hash[:16]}...`",
    f"- Fold plan: train {train_days:.1f}d / test {test_days:.1f}d x 3 folds",
    f"- Fees: maker {maker_fee} ({CONNECTOR}), dead-zone floor {4 * maker_fee:.4f}",
    f"- Fund: {FUND_USD} quote, quote_frac {QUOTE_FRAC}",
    f"- Gate policy: `{gate_policy.describe()}`",
    f"- Objective mode: `{OBJECTIVE_MODE}`",
    f"- Deploy anchor: {deploy_anchor:.6g} (last close {last_close:.6g}, "
    f"divergence {anchor_divergence_pct:.2f}%)",
    "",
    "## Trials",
    "",
    "| total | complete | pruned | failed |",
    "|---|---|---|---|",
    f"| {len(study.trials)} | {len(completed)} | {len(pruned)} | {len(failed)} |",
    "",
    "## Top 10",
    "",
    "| trial | objective | pnl_med % | endinv_med % | cons_med | trades_mo_med | min_rung_touches | gate viol |",
    "|---|---|---|---|---|---|---|---|",
]
for _t in ranked[:10]:
    _u = _t.user_attrs
    _tm = _u.get("trades_per_month_median")
    _mt = _u.get("min_rung_touches")
    _lines.append(
        f"| {_t.number} | {_t.value:.2f} | {_u.get('pnl_pct_median', float('nan')):.2f} "
        f"| {_u.get('endinv_pct_median', float('nan')):.1f} "
        f"| {_u.get('cons_score_median', float('nan')):.1f} "
        f"| {(f'{_tm:.1f}' if _tm is not None else 'n/a')} "
        f"| {(_mt if _mt is not None else 'n/a')} "
        f"| {_u.get('gate_violations', '?')} |")

def _fmt(v, spec=".1f"):
    """n/a for missing values — trials persisted by OLDER notebook versions
    may lack newer fold_detail keys (never crash the report)."""
    return format(v, spec) if isinstance(v, (int, float)) else "n/a"


def _fold_table(rows):
    out = [
        "| fold | score | ann % | cons ann % | endinv % | maxdd % | fills | trades/mo |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for _r in rows:
        if "violation" in _r:
            out.append(f"| {_r['fold']} | — | — | — | — | — | {_r['violation']} | — |")
            continue
        out.append(
            f"| {_r['fold']} | {_fmt(_r.get('score_ann_pct'))} | {_fmt(_r.get('ann_pnl_pct'))} "
            f"| {_fmt(_r.get('cons_ann_pct'))} | {_fmt(_r.get('endinv_pct'))} | {_fmt(_r.get('maxdd'))} "
            f"| {_r.get('buy_fills', '?')}b/{_r.get('sell_fills', '?')}s | {_fmt(_r.get('trades_per_month'))} |")
    return out

if incumbent_summary is not None:
    _lines += ["", "## Incumbent benchmark (same policy + objective)", "",
               f"Objective: **{incumbent_summary['objective']:.2f}**, "
               f"gate violations {incumbent_summary['violations']} (reported, never pruned)", ""]
    _lines += _fold_table(incumbent_summary["folds"])

_lines += ["", f"## Best trial (#{best.number}, objective {best.value:.2f})", ""]
_lines += _fold_table(_fold_detail)
_lines += ["", "### Params", "", "```json", json.dumps(best.params, indent=2), "```"]

_touch_rows = [(_r["fold"], _r.get("rung_touches")) for _r in _fold_detail if _r.get("rung_touches")]
if _touch_rows:
    _lines += ["", "### Per-rung train touches (per fold)", ""]
    for _f, _t in _touch_rows:
        _lines.append(f"- fold {_f}: buys {_t['buys']}, sells {_t['sells']}")
_fill_rows = [(_r["fold"], _r.get("rung_fills")) for _r in _fold_detail if _r.get("rung_fills")]
if _fill_rows:
    _lines += ["", "### Per-rung fills (per fold)", ""]
    for _f, _t in _fill_rows:
        _lines.append(f"- fold {_f}: buys {_t['buys']}, sells {_t['sells']}")

if SEARCH_MODE == "refine_incumbent" and refined_champion is not None:
    _r = refined_champion["rungs"]
    _lines += ["", "## Refinement diff (incumbent → refined)", "",
               f"OOS objective delta: {refined_champion['trial'].value - incumbent_summary['objective']:+.2f}",
               "",
               "| side | # | old price | new price | old wt | new wt |",
               "|---|---|---|---|---|---|"]
    _inc_b = sorted(incumbent["buy_prices"], reverse=True)
    _inc_s = sorted(incumbent["sell_prices"])
    for i, (op, np_, ow, nw) in enumerate(zip(_inc_b, _r["buys"], incumbent["buy_weights"], _r["buy_weights"])):
        _lines.append(f"| buy | {i} | {op} | {np_} | {ow} | {nw:.4f} |")
    for i, (op, np_, ow, nw) in enumerate(zip(_inc_s, _r["sells"], incumbent["sell_weights"], _r["sell_weights"])):
        _lines.append(f"| sell | {i} | {op} | {np_} | {ow} | {nw:.4f} |")

_lines += ["", "## Export", ""]
if yaml_path is not None:
    _lines += [
        f"Export: `{yaml_path}` (validated: {vr.valid})",
        "",
        "### Exported ladder (rebuilt at the deploy anchor)",
        "",
        "```json",
        json.dumps({
            "deploy_anchor": deploy_anchor,
            "buys": [float(x) for x in export_rungs.buys],
            "sells": [float(x) for x in export_rungs.sells],
            "buy_weights": [float(x) for x in export_rungs.buy_weights],
            "sell_weights": [float(x) for x in export_rungs.sell_weights],
        }, indent=2),
        "```",
    ]
    if vr.warnings:
        _lines += ["", "Warnings:"] + [f"- {w}" for w in vr.warnings]
else:
    _lines += [f"**NO EXPORT WRITTEN** — {export_failed_reason}"]

_lines += [
    "",
    "## Phase A caveats",
    "",
    "- No proceeds recycling; static per-rung quantities (Phase B).",
    "- `executor_refresh_time` not modeled (Phase B event-level sim).",
    "- Conservative (stress) scores gate only in accumulate_ok mode; otherwise informational.",
]
report_path = out_dir / f"{TRADING_PAIR}_{INTERVAL}_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(_lines), encoding="utf-8")
print(f"report: {report_path}")

KeyError: 'ann_pnl_pct'

## Notes

- **DASH-USDT / SUN-USDT** still have no lake candles — the preflight aborts
  until the ingester backfills. Kraken `XMR-USD` remains preflight-gated.
- Live incumbents: `configs/incumbents/<connector>__<pair>.yml` (all five
  live configs are checked in; drop future live YAMLs verbatim — CSV rung
  fields are parsed).
- `accumulate_ok` waives the endinv gate — an explicit operator decision for
  accumulation-tolerant pairs (XMR). The exported YAML records the policy.